# Fine-tuning a cross-encoder reranker for the Telugu Gita

The stock reranker (`bge-reranker-v2-m3`) **hurt** quality on this corpus:
NDCG@10 0.395 -> 0.382, and it damaged exactly the query types our fine-tuned
retriever had learned (thematic, tenglish). A reranker only helps if it is
better at the task than the retriever it reorders.

So we teach it the domain. The headroom is real: **recall@50 = 0.810** against
recall@10 = 0.540, so the right verse is already retrieved for 81% of queries
and merely ranked too low.

**Runtime -> Change runtime type -> T4 GPU** first.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
%pip install -q -U sentence-transformers datasets accelerate


## 1. Upload `gita_ce_data.zip`


In [ ]:
from google.colab import files
import zipfile, os, json
up = files.upload()
for n in up:
    if n.endswith('.zip'): zipfile.ZipFile(n).extractall('.'); print('unzipped', n)

def jl(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
train = jl('train_ce.jsonl')
heldc = jl('held_ce_eval.jsonl')
print('train rows      :', len(train))
print('held eval rows  :', len(heldc))
print('pairs per epoch :', sum(1+len(r['negatives']) for r in train))

tr_pos = {r['positive_id'] for r in train}
ev_pos = {r['positive_id'] for r in heldc}
print('CONTAMINATION   :', len(tr_pos & ev_pos), ' <- must be 0')
assert not (tr_pos & ev_pos)


## 2. Eval, written by hand

Deliberately not using a built-in evaluator: the API has churned across
versions and this is 20 lines. Each row already holds the retriever's frozen
top-50, so we measure the only thing a reranker can affect - the ordering of
those 50.


In [ ]:
import numpy as np, math
from collections import defaultdict

def score_ranking(rows, rank_fn, k=10):
    agg = defaultdict(lambda: {'n':0,'hit':0,'ndcg':0.0,'ranks':[]})
    tot = {'n':0,'hit':0,'ndcg':0.0}
    for r in rows:
        ranked = rank_fn(r)
        gold = r['positive_id']
        rank = ranked.index(gold)+1 if gold in ranked else None
        nd = 1.0/math.log2(rank+1) if (rank and rank<=k) else 0.0
        hit = 1 if (rank and rank<=k) else 0
        for d in (tot, agg[r['kind']]):
            d['n']+=1; d['hit']+=hit; d['ndcg']+=nd
        if rank: agg[r['kind']]['ranks'].append(rank)
    out = {'recall@10': tot['hit']/tot['n'], 'ndcg@10': tot['ndcg']/tot['n'], 'per':{}}
    for kind,v in agg.items():
        out['per'][kind] = {'n':v['n'], 'recall@10':v['hit']/v['n'], 'ndcg@10':v['ndcg']/v['n'],
                            'median_rank': sorted(v['ranks'])[len(v['ranks'])//2] if v['ranks'] else None}
    return out

def show(tag, res):
    print(f"{tag:<26} recall@10={res['recall@10']:.3f}  NDCG@10={res['ndcg@10']:.3f}")
    for kind,v in sorted(res['per'].items()):
        print(f"    {kind:<9} n={v['n']:<4} recall@10={v['recall@10']:.3f} "
              f"NDCG@10={v['ndcg@10']:.3f} median={v['median_rank']}")

def ce_rank_fn(model, batch_size=128):
    def f(r):
        s = model.predict([(r['query'], c) for c in r['candidates']],
                          batch_size=batch_size, show_progress_bar=False)
        return [r['candidate_ids'][i] for i in np.argsort(-np.asarray(s))]
    return f

# Measured on the real data: longest (query, doc) pair is 146 tokens.
# 512 wastes ~3.5x the memory for nothing - attention cost is quadratic in
# sequence length, and that was most of the OOM.
MAX_LEN = 192

ceiling = sum(1 for r in heldc if r['positive_id'] in r['candidate_ids'])/len(heldc)
print(f'reranker ceiling (gold present in top-50): {ceiling:.3f}')

import gc, torch
def free_gpu(*objs):
    """`del` alone does NOT release CUDA memory - the allocator keeps the
    blocks cached. Without this the stock reranker stays resident (~2.3 GB)
    and the trainer OOMs later for no visible reason."""
    for o in objs:
        try: o.model.cpu()
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

def gpu_report(tag):
    a = torch.cuda.memory_allocated()/2**30; r = torch.cuda.memory_reserved()/2**30
    print(f'[gpu] {tag:<24} allocated={a:.2f} GiB  reserved={r:.2f} GiB')


## 3. Two baselines

The retriever's own ordering, and the stock reranker we already know degrades it.


In [ ]:
base = score_ranking(heldc, lambda r: r['candidate_ids'])
show('retriever (no rerank)', base)

from sentence_transformers import CrossEncoder
gpu_report('before stock')
stock = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=MAX_LEN)
stock_res = score_ranking(heldc, ce_rank_fn(stock))
show('stock reranker', stock_res)

free_gpu(stock); del stock
gpu_report('after freeing stock')   # should be ~0.00


## 4. Training data

Listwise if the installed version offers it, pointwise otherwise. Listwise
compares candidates against each other (the contrastive idea, with a
cross-attention scoring function); pointwise judges each pair alone and is a
weaker but very robust fallback.


In [ ]:
from datasets import Dataset
import sentence_transformers.cross_encoder.losses as ce_losses

have = dir(ce_losses)
LISTWISE = next((n for n in ['LambdaLoss','ListNetLoss','PListMLELoss'] if n in have), None)
print('available listwise losses:', [n for n in have if 'Loss' in n])
print('chosen:', LISTWISE or 'BinaryCrossEntropyLoss (pointwise fallback)')

if LISTWISE:
    ds = Dataset.from_dict({
        'query':  [r['query'] for r in train],
        'docs':   [[r['positive']] + r['negatives'] for r in train],
        'labels': [[1] + [0]*len(r['negatives']) for r in train],
    })
else:
    q, d, y = [], [], []
    for r in train:
        q.append(r['query']); d.append(r['positive']); y.append(1.0)
        for n in r['negatives']:
            q.append(r['query']); d.append(n); y.append(0.0)
    ds = Dataset.from_dict({'query': q, 'doc': d, 'label': y})
print(ds)


## 5. Train

Batch size means something different here than for the bi-encoder. A
cross-encoder has no embeddings to reuse, so there are no free in-batch
negatives - every negative is one you supplied and paid a forward pass for.
Batch size is purely a memory/throughput knob now, not the quality lever.


In [ ]:
import os, gc, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect(); torch.cuda.empty_cache()
gpu_report('before training')

from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments

BASE = 'BAAI/bge-reranker-v2-m3'   # -> 'BAAI/bge-reranker-base' (278M) if still OOM
model = CrossEncoder(BASE, num_labels=1, max_length=MAX_LEN)
loss  = getattr(ce_losses, LISTWISE)(model) if LISTWISE else ce_losses.BinaryCrossEntropyLoss(model)

# A listwise 'row' is a GROUP of 9 docs, so batch=2 is really 18 sequences.
# Batch size is not the quality lever on a cross-encoder (negatives come from
# the file, not the batch), so shrinking it is free. Adafactor replaces AdamW's
# two fp32 moment tensors (~4.5 GB at 568M params) with a factored estimate.
BS, ACC = (2, 8) if LISTWISE else (16, 1)

args = CrossEncoderTrainingArguments(
    output_dir='bge-reranker-gita',
    num_train_epochs=2,
    per_device_train_batch_size=BS,
    gradient_accumulation_steps=ACC,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    fp16=True,
    optim='adafactor',
    logging_steps=25,
    save_strategy='no',
    report_to='none',
)
print(f'{BASE}  max_length={MAX_LEN}  batch={BS}x{ACC}  loss={LISTWISE or "BCE"}  optim=adafactor')
CrossEncoderTrainer(model=model, args=args, train_dataset=ds, loss=loss).train()


## 6. Did it work?


In [ ]:
tuned = score_ranking(heldc, ce_rank_fn(model))
print()
show('retriever (no rerank)', base)
show('stock reranker', stock_res)
show('FINE-TUNED reranker', tuned)
print()
print(f"ceiling (recall@50)        {ceiling:.3f}")
gap = ceiling - base['recall@10']
got = tuned['recall@10'] - base['recall@10']
print(f"headroom {gap:+.3f} | captured {got:+.3f} | {100*got/gap if gap else 0:.0f}% of it")


## 7. Save


In [ ]:
model.save_pretrained('bge-reranker-gita-final')
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/telugu-rag'
!cp -r bge-reranker-gita-final '/content/drive/MyDrive/telugu-rag/'
!du -sh '/content/drive/MyDrive/telugu-rag/bge-reranker-gita-final'
